# 01 - Run zero-shot baselines on FLORES-200
Runs on the Colab T4. Gets the latest code from GitHub, then runs `src/translate.py` with a config.

Edit code on your laptop -> push to GitHub -> re-run cell 2 here to pull the change.

In [ ]:
# Cell 1: GPU check + Hugging Face login
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch runtime to T4")

from huggingface_hub import login, whoami
try:
    whoami()
    print("Already logged in to Hugging Face")
except Exception:
    from getpass import getpass
    login(token=getpass("Paste your Hugging Face token: "))

In [ ]:
# Cell 2: get (or update) the code from GitHub and install packages
import os
REPO_URL = "https://github.com/prefierolasoledad/TourMT-Hi.git"
WORKDIR = "/content/tourmt-hi"

if not os.path.exists(WORKDIR):
    !git clone {REPO_URL} {WORKDIR}
else:
    # Make the Colab copy exactly match GitHub (GitHub is the master copy;
    # files changed only here on Colab are replaced by GitHub's version).
    !git -C {WORKDIR} fetch -q origin && git -C {WORKDIR} reset -q --hard origin/main
%cd {WORKDIR}
# Install packages; show only errors and the final summary (so failures are not hidden)
!pip install -r requirements.txt 2>&1 | grep -iE "error|failed|successfully installed" || true

# Check the packages we need actually import
import importlib
for pkg in ["transformers", "datasets", "sacrebleu"]:
    try:
        m = importlib.import_module(pkg)
        print(f"OK      {pkg} {getattr(m, '__version__', '')}")
    except Exception as e:
        print(f"MISSING {pkg}: {type(e).__name__}: {e}")
!git log -1 --oneline

In [ ]:
# Cell 2b: separate Python 3.11 environment for evaluation + IndicTrans2 (both need transformers<5)
# Takes 1-2 minutes; needed once per Colab session. See requirements-eval.txt.
import os
if not os.path.exists("/content/venv-eval/bin/python"):
    !pip install -q uv
    !uv venv -q /content/venv-eval --python 3.11
# Always run: installs anything new in requirements-eval.txt (fast if nothing changed)
!uv pip install -q --python /content/venv-eval/bin/python -r requirements-eval.txt
# --no-deps on purpose: its dependency is a broken fork of indicnlp (see requirements-eval.txt)
!uv pip install -q --python /content/venv-eval/bin/python --no-deps IndicTransToolkit==1.1.1
!/content/venv-eval/bin/python -c "import sys; sys.path.insert(0, 'src'); import comet, sacrebleu, torch, transformers, datasets; from IndicTransToolkit.processor import IndicProcessor; from common import check_indicnlp_normalizer; check_indicnlp_normalizer(); from importlib.metadata import version; print('eval env OK | comet', version('unbabel-comet'), '| transformers', transformers.__version__, '| GPU', torch.cuda.is_available())"

In [ ]:
# Cell 3: smoke test - 20 sentences only (about a minute)
!python src/translate.py --config configs/nllb600m_flores-devtest_en-hi.yaml --limit 20

In [ ]:
# Cell 4: full NLLB-600M runs on FLORES devtest (1012 sentences each)
for cfg in ["nllb600m_flores-devtest_en-hi", "nllb600m_flores-devtest_hi-en"]:
    !python src/translate.py --config configs/{cfg}.yaml

In [ ]:
# Cell 4b: IndicTrans2-200M runs on FLORES devtest (uses the Python 3.11 environment from Cell 2b)
# The models are gated: accept the terms once on both pages while logged in to Hugging Face:
#   https://huggingface.co/ai4bharat/indictrans2-en-indic-dist-200M
#   https://huggingface.co/ai4bharat/indictrans2-indic-en-dist-200M
PY = "/content/venv-eval/bin/python"
!{PY} src/translate.py --config configs/it2-200m_flores-devtest_en-hi.yaml --limit 20   # quick test first
for cfg in ["it2-200m_flores-devtest_en-hi", "it2-200m_flores-devtest_hi-en"]:
    !{PY} src/translate.py --config configs/{cfg}.yaml

In [ ]:
# Cell 4c: Maithili (English -> Maithili) runs on FLORES devtest, both models
!python src/translate.py --config configs/nllb600m_flores-devtest_en-mai.yaml                     # NLLB: Colab's Python
!/content/venv-eval/bin/python src/translate.py --config configs/it2-200m_flores-devtest_en-mai.yaml  # IndicTrans2: Python 3.11 env

In [ ]:
# Cell 4d: how well does GlotLID separate Hindi from Maithili? (check before trusting off-target rates)
# First run downloads the GlotLID model (783 MB).
!/content/venv-eval/bin/python src/check_lid.py --split devtest

In [52]:
# Cell 5: score ALL FLORES runs (chrF++, BLEU, COMET, CometKiwi) -> results/results.csv
# Uses the separate evaluation environment from Cell 2b. --skip-scored: runs that already
# have scores.json are not scored again (delete a run's scores.json to force it).
# CometKiwi is gated: accept its terms on https://huggingface.co/Unbabel/wmt22-cometkiwi-da first.
!/content/venv-eval/bin/python src/evaluate.py --skip-scored --runs $(ls -d outputs/*_flores-devtest_* | grep -v _limit)
!cut -d, -f1,7-12 results/results.csv | column -s, -t

Already scored, skipping: ['it2-200m_flores-devtest_en-hi', 'it2-200m_flores-devtest_en-mai', 'it2-200m_flores-devtest_hi-en', 'nllb600m_flores-devtest_en-hi', 'nllb600m_flores-devtest_hi-en']

== nllb600m_flores-devtest_en-mai (1012 sentences, normalisation: indicnlp-hi (nukta + alternative forms))
  chrF++ 43.11   BLEU 12.99
/content/venv-eval/lib/python3.11/site-packages/torchmetrics/utilities/imports.py:23: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution

Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.6.6. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../../root/.cache/huggingface/hub/models--Unbabel--wmt22-comet-da/snapshots/2760

In [53]:
# Cell 6: download the outputs to your laptop as one zip file
!cd {WORKDIR} && zip -qr /content/outputs.zip outputs results -x '*_limit*'
print("Created /content/outputs.zip - download it and unzip it into the project folder on your laptop")

Created /content/outputs.zip - download it and unzip it into the project folder on your laptop


In [54]:
# Cell 7: put the zip into this notebook's output so it can be rebuilt on the laptop
import base64
data = open("/content/outputs.zip", "rb").read()
print(f"zip size: {len(data)/1024:.0f} KB")
print("===ZIP-START===")
print(base64.b64encode(data).decode())
print("===ZIP-END===")

zip size: 1227 KB
===ZIP-START===
...
===ZIP-END===
